# Pneumonia Detection & Localization — Member 2
## Model Training & Evaluation Guide

**Member 2 Responsibilities:**
- Train DenseNet121 binary classifier on `train.csv` and `val.csv` using `BCEWithLogitsLoss` and `Adam`.
- Track validation ROC-AUC and save the best checkpoint (`best_model.pth`) directly to Google Drive.
- Evaluate once on held-out `test.csv` after model selection is complete.
- Report Accuracy, Precision, Recall, F1-Score, ROC-AUC, Confusion Matrix, and ROC Curve.
- Export result artifacts to Google Drive rather than GitHub.

### Step 1: Check GPU Acceleration
Ensure your Colab runtime is set to GPU (**Runtime → Change runtime type → T4 GPU**).

In [ ]:
!nvidia-smi

### Step 2: Mount Google Drive
Mount Google Drive to access shared metadata, image datasets, and save checkpoints.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 3: Clone Repository and Switch to `feature/model`

In [ ]:
%cd /content
!rm -rf Pneumonia_Detection_Localization
!git clone https://github.com/Tanujn06/Pneumonia_Detection_Localization.git
%cd /content/Pneumonia_Detection_Localization
!git fetch origin
!git switch -c feature/model --track origin/feature/model || !git checkout feature/model
!git status

### Step 4: Install Dependencies & Extract Images
Extract training images into Colab's fast local SSD storage (`/content/rsna_train_images/`).

In [ ]:
!pip install -q pydicom

# Create persistent output folders in Google Drive
!mkdir -p /content/drive/MyDrive/Pneumonia_Project/checkpoints
!mkdir -p /content/drive/MyDrive/Pneumonia_Project/results
!mkdir -p /content/rsna_train_images

import os
if not os.path.exists('/content/rsna_train_images/stage_2_train_images'):
    print('Extracting images from Drive to local Colab SSD (takes ~2 minutes)...')
    !unzip -q '/content/drive/MyDrive/dataset/stage_2_train_images.zip' -d /content/rsna_train_images/
    print('Extraction finished!')
else:
    print('Training images already extracted in local runtime.')

### Step 5: Train DenseNet121 Binary Classifier
- Model selection is tracked on **Validation ROC-AUC**.
- Checkpoints are saved directly to Google Drive (`best_model.pth`).

In [ ]:
!python rc/src/train.py \
    --train_csv '/content/drive/MyDrive/Pneumonia_Project/metadata/train.csv' \
    --val_csv '/content/drive/MyDrive/Pneumonia_Project/metadata/val.csv' \
    --img_dir '/content/rsna_train_images/stage_2_train_images/' \
    --checkpoint_dir '/content/drive/MyDrive/Pneumonia_Project/checkpoints' \
    --epochs 15 \
    --batch_size 32 \
    --lr 0.0001 \
    --num_workers 2

### Step 5b: Confirm Saved Checkpoint and Training Logs
Verify that  is saved on Google Drive, confirm its saved epoch and validation AUC, and inspect the final 15-epoch training logs before touching the test set.

In [ ]:
# 1. Verify saved best_model.pth checkpoint
!python rc/src/verify_checkpoint.py \n    --checkpoint '/content/drive/MyDrive/Pneumonia_Project/checkpoints/best_model.pth'

# 2. Display the full training logs across all 15 epochs
with open('/content/drive/MyDrive/Pneumonia_Project/checkpoints/training_logs.txt', 'r') as f:
    print(f.read())

### Step 6: Evaluate Once on Held-out `test.csv`
Evaluates `best_model.pth` on the test set and calculates Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix, and ROC Curve.

In [ ]:
!python rc/src/evaluate.py \
    --test_csv '/content/drive/MyDrive/Pneumonia_Project/metadata/test.csv' \
    --img_dir '/content/rsna_train_images/stage_2_train_images/' \
    --checkpoint '/content/drive/MyDrive/Pneumonia_Project/checkpoints/best_model.pth' \
    --results_dir '/content/drive/MyDrive/Pneumonia_Project/results' \
    --batch_size 32

### Step 7: Display Test Metrics & Plots
Review the generated evaluation artifacts saved in Google Drive.

In [ ]:
import json
from IPython.display import Image, display

results_dir = '/content/drive/MyDrive/Pneumonia_Project/results'

with open(f'{results_dir}/test_metrics.json', 'r') as f:
    metrics = json.load(f)

print('='*40)
print('      HELD-OUT TEST RESULTS')
print('='*40)
for k, v in metrics.items():
    if k != 'confusion_matrix':
        print(f'{k:15s}: {v:.4f}')
print('='*40)

print('\n--- Confusion Matrix ---')
display(Image(filename=f'{results_dir}/confusion_matrix.png'))

print('\n--- ROC Curve ---')
display(Image(filename=f'{results_dir}/roc_curve.png'))